## South Africa Public Procurement Intelligence System

### Complete Pipeline Notebook

#### 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from config import DB_CONFIG, CATEGORY_MAP
import mysql.connector



# Clear cache and add project root
if 'config' in sys.modules:
    del sys.modules['config']

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import config
from config import DB_CONFIG

print("✅ DB_CONFIG imported successfully")
print(f"Host: {DB_CONFIG['host']}, Database: {DB_CONFIG['database']}")

✅ DB_CONFIG imported successfully
Host: localhost, Database: procurement_intelligence


In [2]:
# Test config import
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Clear any cached config
if 'config' in sys.modules:
    del sys.modules['config']

print("✅ config imported successfully")
print("DB_CONFIG:", DB_CONFIG)

✅ config imported successfully
DB_CONFIG: {'host': 'localhost', 'port': 3306, 'user': 'root', 'password': 'toni_$k8', 'database': 'procurement_intelligence', 'charset': 'utf8mb4'}


In [ ]:
# Test direct MySQL connection


project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))


print("Attempting connection with:")
print(f"Host: {DB_CONFIG['host']}, User: {DB_CONFIG['user']}, DB: {DB_CONFIG['database']}")

try:
    conn = mysql.connector.connect(**DB_CONFIG)
    print("✅ MySQL connection successful!")
    conn.close()
except Exception as e:
    print(f"❌ Connection failed: {e}")
    

Attempting connection with:
Host: localhost, User: root, DB: procurement_intelligence
✅ MySQL connection successful!


#### 2. Load Data from MySQL

In [ ]:
# Cell: Load full dataset from MySQL – all awards (no contract restriction)

# 1. Load awards with supplier names and tender info
AWARDS_QUERY = """
SELECT
    a.id AS award_id,
    a.value_amount AS award_value,
    a.main_ocid,
    s.name AS supplier_name,
    m.ocid,
    m.buyer_name,
    m.tender_province AS province,
    m.tender_mainprocurementcategory AS category,
    m.tender_procurementmethod AS method,
    m.date AS tender_date,
    YEAR(m.date) AS year,
    MONTH(m.date) AS month,
    QUARTER(m.date) AS quarter
FROM awards_staging a
LEFT JOIN awards_suppliers_staging s ON a.id = s.awards_id
LEFT JOIN main_staging m ON a.main_ocid = m.ocid
WHERE a.value_amount > 0
  AND m.tender_province IS NOT NULL
"""

conn = mysql.connector.connect(**DB_CONFIG)
df_raw = pd.read_sql(AWARDS_QUERY, conn, parse_dates=["tender_date"])

# 2. Load contract details separately
CONTRACTS_QUERY = """
SELECT
    main_ocid,
    value_amount AS contract_value,
    period_durationindays AS duration_days
FROM contracts_staging
WHERE value_amount > 0
"""
contracts_df = pd.read_sql(CONTRACTS_QUERY, conn)
conn.close()

# 3. Merge contracts onto awards
df_raw = df_raw.merge(contracts_df, on='main_ocid', how='left')

# 4. Use contract value if present, else award value
df_raw['contract_value'] = np.where(
    df_raw['contract_value'].notna() & (df_raw['contract_value'] > 0),
    df_raw['contract_value'],
    df_raw['award_value']
)

# 5. Clean text fields
df_raw['category'] = df_raw['category'].str.strip().str.lower().map(CATEGORY_MAP).fillna('Unknown')
df_raw['method']   = df_raw['method'].str.strip().str.lower().fillna('unknown')
df_raw['province'] = df_raw['province'].str.strip().str.title()
df_raw['duration_days'] = pd.to_numeric(df_raw['duration_days'], errors='coerce').fillna(0)

print(f"✅ Loaded {len(df_raw):,} awards with tender & supplier info")
df_raw.head()

C:\Users\kalol\AppData\Local\Temp\ipykernel_9748\3609371527.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_raw = pd.read_sql(AWARDS_QUERY, conn, parse_dates=["tender_date"])


✅ Loaded 5,679 awards with tender & supplier info


C:\Users\kalol\AppData\Local\Temp\ipykernel_9748\3609371527.py:42: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  contracts_df = pd.read_sql(CONTRACTS_QUERY, conn)


,award_id,award_value,main_ocid,supplier_name,ocid,buyer_name,province,category,method,tender_date,year,month,quarter,contract_value,duration_days
0,37013,5977550.0,ocds-9t57fa-100504,(AFRICAN WEALTH EMPOWERMENT NETWORK (AWEN)),ocds-9t57fa-100504,The Playhouse Company,Kwazulu-Natal,Works,open,2024-09-02,2024,9,3,5977550.0,0.0
1,38439,31521960.0,ocds-9t57fa-100557,(IMSIMBI TRAINING),ocds-9t57fa-100557,South African Civil Aviation Authority,Gauteng,Unknown,open,2024-09-03,2024,9,3,31521960.0,0.0
2,41106,473352120.0,ocds-9t57fa-100587,(MSEETO PROJECTS),ocds-9t57fa-100587,Eastern Cape - Social Development and Special ...,Eastern Cape,Services,open,2024-09-03,2024,9,3,4733521.0,1094.0
3,40261,722758655.0,ocds-9t57fa-100636,(IN2IT),ocds-9t57fa-100636,Road Accident Fund,Gauteng,Unknown,open,2024-09-03,2024,9,3,722758655.0,0.0
4,39101,19205000.0,ocds-9t57fa-100649,(PULE AND VIC CONSTRUCTION AND PROJECTS),ocds-9t57fa-100649,Justice & Constitutional Development,Mpumalanga,Works,open,2024-09-03,2024,9,3,19205000.0,0.0


In [ ]:
# Filter to high-quality records + add powerful features

# Keep awards that have a real contract (duration > 0) OR are large (>R500k)
df = df_raw.copy()
df['has_contract'] = (df['duration_days'] > 0).astype(int)
df_filtered = df[(df['has_contract'] == 1) | (df['contract_value'] > 500_000)].copy()

print(f"After filtering: {len(df_filtered):,} rows (from {len(df):,})")

# ----- Recore features on filtered data -----
df_filtered['log_value'] = np.log1p(df_filtered['contract_value'])
df_filtered['tender_date'] = pd.to_datetime(df_filtered['tender_date'], errors='coerce')
df_filtered['year']   = df_filtered['tender_date'].dt.year
df_filtered['month']  = df_filtered['tender_date'].dt.month
df_filtered['quarter']= df_filtered['tender_date'].dt.quarter
df_filtered['is_peak_month'] = (df_filtered['month'] == 8).astype(int)
df_filtered['is_year_end_quarter'] = (df_filtered['quarter'] == 4).astype(int)

# Duration
df_filtered['duration_days'] = pd.to_numeric(df_filtered['duration_days'], errors='coerce').fillna(0)
df_filtered['duration_years'] = df_filtered['duration_days'] / 365.25
df_filtered['duration_band'] = pd.cut(
    df_filtered['duration_days'],
    bins=[0, 90, 180, 365, 730, 1095, np.inf],
    labels=["<3mo", "3-6mo", "6-12mo", "1-2yr", "2-3yr", ">3yr"],
    right=True
)

# Interaction
df_filtered['province_quarter'] = df_filtered['province'] + "_Q" + df_filtered['quarter'].astype(str)
df_filtered['prov_cat_q'] = df_filtered['province'] + "_" + df_filtered['category'] + "_Q" + df_filtered['quarter'].astype(str)

# Supplier history
df_filtered['supplier_prior_awards'] = df_filtered.groupby('supplier_name')['award_id'].transform('count')
df_filtered['is_new_supplier'] = (df_filtered['supplier_prior_awards'] == 1).astype(int)

# Buyer aggregates
df_filtered['buyer_mean'] = df_filtered.groupby('buyer_name')['contract_value'].transform('mean')
df_filtered['buyer_std']  = df_filtered.groupby('buyer_name')['contract_value'].transform('std')
df_filtered['buyer_count'] = df_filtered.groupby('buyer_name')['ocid'].transform('nunique')

# Time features (days since supplier's last award)
df_filtered = df_filtered.sort_values(['supplier_name','tender_date'])
df_filtered['days_since_supplier_last'] = df_filtered.groupby('supplier_name')['tender_date'].diff().dt.days.fillna(0)

# Outlier capping (99.5th percentile)
cap = df_filtered['contract_value'].quantile(0.995)
df_filtered['contract_value_capped'] = df_filtered['contract_value'].clip(upper=cap)
df_filtered['log_value'] = np.log1p(df_filtered['contract_value_capped'])

print(f"✅ Filtered & engineered: {len(df_filtered)} rows, {df_filtered.shape[1]} columns")
df_raw = df_filtered   # continue with this variable name for compatibility

After filtering: 4,753 rows (from 5,679)
✅ Filtered & engineered: 4753 rows, 30 columns


In [6]:
# Quick data summary
print("=== Data Summary ===")
print(f"Total rows: {len(df_raw):,}")
print(f"Unique tenders (ocid): {df_raw['ocid'].nunique():,}")
print(f"Unique suppliers: {df_raw['supplier_name'].nunique():,}")
print(f"Date range: {df_raw['tender_date'].min()} to {df_raw['tender_date'].max()}")
print("\n=== Provinces ===")
print(df_raw['province'].value_counts())
print("\n=== Categories ===")
print(df_raw['category'].value_counts())

=== Data Summary ===
Total rows: 4,753
Unique tenders (ocid): 4,615
Unique suppliers: 4,018
Date range: 2017-03-31 00:00:00 to 2026-03-10 00:00:00

=== Provinces ===
province
Gauteng          1056
Kwazulu-Natal     827
Western Cape      740
Eastern Cape      667
National          635
Limpopo           355
Mpumalanga        164
Free State        120
North West        106
Northern Cape      83
Name: count, dtype: int64

=== Categories ===
category
Services    1933
Works       1090
Unknown      977
Goods        753
Name: count, dtype: int64


#### 3. Run Anomaly Detection

In [7]:
from models.anomaly_detector import run_full_anomaly_pipeline, anomaly_summary_report
from models.opportunity_matrix import build_opportunity_matrix, get_top_opportunities, save_matrix

df_raw = run_full_anomaly_pipeline(df_raw)
summary = anomaly_summary_report(df_raw)
print(f"Flagged: {summary['total_flagged']} ({summary['flag_rate_pct']}%)")

Flagged: 1951 (41.05%)


#### 4. Build Opportunity Matrix

In [8]:
matrix = build_opportunity_matrix(df_raw)
print(f"Matrix built with {len(matrix)} cells")
get_top_opportunities(matrix, top_n=10)

Matrix built with 156 cells


,value_rank,province,category,quarter_label,contract_count,median_value_fmt,total_value_fmt,reliability_tier,opportunity_score,pct_of_national_spend,gauteng_flag
0,1,Mpumalanga,Works,Q3 (Jul-Sep),12,R1.77B,R22.30B,High Confidence,25.5525,0.711,0
1,2,Eastern Cape,Works,Q3 (Jul-Sep),98,R322.10M,R168.17B,High Confidence,23.5084,5.365,0
2,3,Western Cape,Works,Q3 (Jul-Sep),45,R95.35M,R130.45B,High Confidence,22.0476,4.161,0
3,4,Limpopo,Works,Q3 (Jul-Sep),22,R89.68M,R19.39B,High Confidence,21.9741,0.618,0
4,5,Kwazulu-Natal,Works,Q3 (Jul-Sep),115,R63.51M,R18.29B,High Confidence,21.5601,0.583,0
5,6,National,Works,Q3 (Jul-Sep),18,R49.45M,R4.19B,High Confidence,21.2597,0.134,0
6,7,Gauteng,Works,Q3 (Jul-Sep),48,R49.02M,R30.89B,High Confidence,21.2492,0.985,1
7,8,North West,Works,Q3 (Jul-Sep),11,R44.06M,R4.61B,High Confidence,21.1213,0.147,0
8,9,Limpopo,Unknown,Q3 (Jul-Sep),18,R40.95M,R41.45B,High Confidence,21.0334,1.322,0
9,10,Limpopo,Services,Q3 (Jul-Sep),39,R30.00M,R12.01B,High Confidence,20.6600,0.383,0


#### 5. Train ML Models

In [ ]:
# %%  Final Improved Model Training – Safe Features, Proper CV, No Leakage
from sklearn.ensemble import StackingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, r2_score
from models.value_forecaster import _build_models
from utils.feature_engineering import SafeLabelEncoder, FrequencyEncoder
from config import RANDOM_STATE, TEST_SIZE, CV_FOLDS, MODEL_DIR
import joblib


df = df_raw.copy()

# Cyclic encoding for month (sin/cos) – avoids treating month as linear
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Cyclic encoding for quarter (capture seasonal cycle more directly)
df['quarter_sin'] = np.sin(2 * np.pi * df['quarter'] / 4)
df['quarter_cos'] = np.cos(2 * np.pi * df['quarter'] / 4)

# Buyer × category interaction count (how many times this buyer bought this category)
df['buyer_cat_count'] = df.groupby(['buyer_name','category'])['ocid'].transform('nunique')
# Supplier × category interaction count
df['supplier_cat_count'] = df.groupby(['supplier_name','category'])['award_id'].transform('count')

# These new columns need to be added to the feature set
extra_num_cols = ['month_sin','month_cos','quarter_sin','quarter_cos',
                'buyer_cat_count','supplier_cat_count']

# ---------- 2. Build feature matrix (extended) ----------
from utils.feature_engineering import CAT_COLS, NUM_COLS, INTERACTION_COLS

# Original numeric columns (already in the dataset)
original_num_cols = [col for col in NUM_COLS if col in df.columns]
# Combine with new ones
all_num_cols = original_num_cols + extra_num_cols

# Ensure all required columns exist
for col in CAT_COLS + all_num_cols + INTERACTION_COLS:
    if col not in df.columns:
        raise ValueError(f"Missing column: {col}")

# Prepare feature dataframe
feat_df = df[CAT_COLS + all_num_cols + INTERACTION_COLS].copy()

# Encode categoricals
enc = SafeLabelEncoder(cols=CAT_COLS)
feat_df[CAT_COLS] = enc.fit_transform(feat_df[CAT_COLS])

# Frequency-encode interactions
freq_enc = FrequencyEncoder(cols=INTERACTION_COLS)
feat_df[INTERACTION_COLS] = freq_enc.fit_transform(feat_df[INTERACTION_COLS])

# Target
y = df['log_value']

# Final X
X = feat_df.fillna(0)
feature_names = list(X.columns)

# ---------- 3. Train/Test split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# ---------- 4. Model definitions ----------
base = _build_models()

# Your tuned Random Forest (best params from earlier GridSearch)
tuned_rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=3,
    max_features=0.5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
base['random_forest'] = tuned_rf

# Stacking with a small Random Forest as meta‑model
stacking = StackingRegressor(
    estimators=[
        ("xgb", base["xgboost"]),
        ("lgb", base["lightgbm"]),
        ("rf", tuned_rf)
    ],
    final_estimator=RandomForestRegressor(n_estimators=50, max_depth=3,
                                        random_state=RANDOM_STATE),
    cv=5,
    n_jobs=-1
)
base['stacking'] = stacking

# ---------- 5. Train & evaluate ----------
kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
results = {}

for name, model in base.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)

    # Cross‑validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='r2', n_jobs=-1)
    cv_r2_mean = round(cv_scores.mean(), 4)
    cv_r2_std  = round(cv_scores.std(), 4)

    # Test metrics
    y_pred_log = model.predict(X_test)
    r2_test = round(r2_score(y_test, y_pred_log), 4)
    mae_log  = round(mean_absolute_error(y_test, y_pred_log), 4)
    y_pred_zar = np.expm1(y_pred_log)
    y_true_zar = np.expm1(y_test)
    mape = round(np.mean(np.abs((y_true_zar - y_pred_zar) / (y_true_zar + 1))) * 100, 2)

    results[name] = {
        'model': model,
        'metrics': {
            'R² (test)': r2_test,
            'CV R² (mean)': cv_r2_mean,
            'CV R² (std)': cv_r2_std,
            'MAE (log)': mae_log,
            'MAPE %': mape,
        }
    }
    # Save model
    joblib.dump({'model': model, 'feature_names': feature_names,
                'label_enc': enc, 'freq_enc': freq_enc},
                MODEL_DIR / f"{name}.pkl")

# ---------- 6. Print results ----------
print("\n" + "=" * 70)
print("  MODEL COMPARISON — Final Safe Feature Set")
print("=" * 70)
rows = [{'Model': name, **m['metrics']} for name, m in results.items()]
df_res = pd.DataFrame(rows).sort_values('R² (test)', ascending=False)
print(df_res.to_string(index=False))
print("=" * 70)

# Feature importance from tuned RF
if hasattr(tuned_rf, 'feature_importances_'):
    fi = pd.Series(tuned_rf.feature_importances_, index=feature_names).sort_values(ascending=False)
    print("\nTop 10 Feature Importances (Random Forest):")
    print(fi.head(10).to_string())

Training ridge...
Training random_forest...
Training xgboost...
Training lightgbm...
Training ensemble...
Training stacking...

  MODEL COMPARISON — Final Safe Feature Set
        Model  R² (test)  CV R² (mean)  CV R² (std)  MAE (log)  MAPE %
     stacking     0.3988        0.3759       0.0208     1.5740  539.44
     ensemble     0.3919        0.3807       0.0239     1.5538  614.88
random_forest     0.3793        0.3619       0.0149     1.6150  537.02
      xgboost     0.3667        0.3549       0.0277     1.5791  677.76
     lightgbm     0.3571        0.3471       0.0320     1.5824  750.02
        ridge     0.1740        0.1510       0.0080     1.9924  700.73

Top 10 Feature Importances (Random Forest):
duration_days       0.201556
buyer_cat_count     0.173032
duration_band       0.138303
year                0.105155
category            0.074167
prov_cat_q          0.062013
province            0.051220
province_quarter    0.049658
month               0.030366
month_sin           0.026

#### 6. Save Outputs

In [12]:
# Cell: Import paths from config
from config import DATA_DIR, MODEL_DIR, REPORT_DIR
from models.opportunity_matrix import save_matrix

# Ensure directories exist
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

print(f"Data dir: {DATA_DIR}")
print(f"Model dir: {MODEL_DIR}")

Data dir: C:\Users\kalol\Documents\SA_Public_Procurement_Intelligence_System\08_procurement_intelligence\data
Model dir: C:\Users\kalol\Documents\SA_Public_Procurement_Intelligence_System\08_procurement_intelligence\models


In [13]:
# Cell: Save outputs
df_raw.to_csv(DATA_DIR / "master_with_anomalies.csv", index=False)
save_matrix(matrix)
print("✅ Data and matrix saved.")

✅ Data and matrix saved.


#### 7. Generate and Save Excel Report

In [16]:
# Generate Excel report
from reports.excel_reporter import generate_excel_report
from models.recommendation_engine import generate_policy_brief
from models.anomaly_detector import anomaly_summary_report

# Compute summary
summary = anomaly_summary_report(df_raw)

# Create the policy brief
brief = generate_policy_brief(df_raw, matrix, summary)

# Generate and save the Excel file
output_path = generate_excel_report(df_raw, matrix, brief)
print(f"✅ Excel report saved to: {output_path}")

✅ Excel report saved to: C:\Users\kalol\Documents\SA_Public_Procurement_Intelligence_System\08_procurement_intelligence\reports\Procurement_Intelligence_Report.xlsx
